# nanoGPT — From-Scratch Decoder-Only Summarizer on CNN/DailyMail

This notebook trains a small decoder-only transformer (~10M non-embedding params, ~26M total) from random weights on CNN/DailyMail using TL;DR-style next-token prediction. It's the **from-scratch baseline** for the comparative analysis against fine-tuned GPT-2.

**Runs unattended on:**
- **Kaggle** (T4×2): auto-uses both GPUs via `torchrun`, ~3-4h.
- **Colab free** (single T4): single GPU, ~6h.

**Just hit "Run all" — no manual steps required.** Internet must be enabled (Kaggle: notebook settings sidebar; Colab: on by default).

**Architecture choice:** 6 layers, 6 heads, n_embd=384, block_size=512, GPT-2 BPE (vocab 50257). Total ~26M params, of which ~7M are in transformer blocks; the GPT-2 BPE embedding table contributes the rest. We keep the GPT-2 tokenizer so ROUGE numbers compare apples-to-apples with the fine-tuned GPT-2 notebook.

## 1. Environment detection + GPU check

In [1]:
import os, sys, subprocess

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ENV = "colab" if IS_COLAB else ("kaggle" if IS_KAGGLE else "local")

WORK_DIR = "/kaggle/working" if IS_KAGGLE else ("/content" if IS_COLAB else os.getcwd())
NANO_DIR = os.path.join(WORK_DIR, "nanoGPT")
OUT_DIR = os.path.join(WORK_DIR, "nano_out")
DATA_DIR = os.path.join(NANO_DIR, "data", "cnn_dailymail")

print(f"Environment: {ENV}")
print(f"Work dir:    {WORK_DIR}")
print(f"nanoGPT dir: {NANO_DIR}")
print(f"Checkpoints: {OUT_DIR}")
print()
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=False)

Environment: colab
Work dir:    /kaggle/working
nanoGPT dir: /kaggle/working/nanoGPT
Checkpoints: /kaggle/working/nano_out

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.105.08
Tesla T4, 15360 MiB, 580.105.08


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], returncode=0)

## 2. Install dependencies

In [2]:
!pip install -q tiktoken datasets transformers tqdm rouge-score

## 3. Clone nanoGPT and create directories

In [3]:
import os, subprocess
if not os.path.exists(NANO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/karpathy/nanoGPT.git", NANO_DIR], check=True)
else:
    print("nanoGPT already cloned.")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print("Ready.")

Cloning into '/kaggle/working/nanoGPT'...


Ready.


## 4. Data preparation

We tokenize each `(article, summary)` pair as:

```
<article>\n\nTL;DR:\n<summary><|endoftext|>
```

then concatenate the entire training set into one `train.bin` (uint16 stream) — the format nanoGPT was built for. The `\n\nTL;DR:\n` prefix is the conditioning signal the model learns: when it sees this prefix at inference, it has learned to continue with a summary.

In [4]:
prepare_code = r"""
import os
import numpy as np
from tqdm import tqdm
import tiktoken
from datasets import load_dataset

ds = load_dataset("cnn_dailymail", "3.0.0")
enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token  # 50256

def format_example(ex):
    text = ex["article"].strip() + "\n\nTL;DR:\n" + ex["highlights"].strip()
    ids = enc.encode_ordinary(text)
    ids.append(EOT)
    return {"ids": ids, "len": len(ids)}

out_dir = os.path.dirname(os.path.abspath(__file__))

split_to_filename = {"train": "train.bin", "validation": "val.bin", "test": "test.bin"}

for split, bin_name in split_to_filename.items():
    bin_path = os.path.join(out_dir, bin_name)
    if os.path.exists(bin_path):
        print(f"{bin_path} exists, skipping.")
        continue
    print(f"Tokenizing {split} ...")
    tokenized = ds[split].map(
        format_example,
        remove_columns=ds[split].column_names,
        desc=f"tokenize {split}",
        num_proc=4,
    )
    total_len = int(np.sum(tokenized["len"], dtype=np.uint64))
    print(f"{split}: {total_len:,} tokens")
    arr = np.memmap(bin_path, dtype=np.uint16, mode="w+", shape=(total_len,))
    idx = 0
    n_shards = 64
    for shard_i in tqdm(range(n_shards), desc=f"writing {bin_name}"):
        shard = tokenized.shard(num_shards=n_shards, index=shard_i, contiguous=True).with_format("numpy")
        arr_shard = np.concatenate(shard["ids"])
        arr[idx : idx + len(arr_shard)] = arr_shard
        idx += len(arr_shard)
    arr.flush()
    print(f"Wrote {bin_path} ({os.path.getsize(bin_path) / 1e6:.1f} MB)")

print("Done.")
"""

with open(os.path.join(DATA_DIR, "prepare.py"), "w") as f:
    f.write(prepare_code)
print(f"Wrote {os.path.join(DATA_DIR, 'prepare.py')}")

Wrote /kaggle/working/nanoGPT/data/cnn_dailymail/prepare.py


Run the prep (downloads CNN/DM ~150MB, tokenizes, writes ~400MB of .bin files; ~10-15 min).

In [5]:
import subprocess, sys
result = subprocess.run([sys.executable, "prepare.py"], cwd=DATA_DIR)
assert result.returncode == 0, "Data prep failed"
print("\nFiles in data dir:")
for f in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, f)
    print(f"  {f:20s} {os.path.getsize(p) / 1e6:>8.1f} MB")

Generating test split: 100%|██████████| 11490/11490 [00:00<00:00, 66626.32 examples/s]


Tokenizing train ...


tokenize train (num_proc=4): 100%|██████████| 287113/287113 [01:17<00:00, 3704.14 examples/s]


train: 270,377,917 tokens


writing train.bin: 100%|██████████| 64/64 [01:44<00:00,  1.63s/it]


...

Wrote /kaggle/working/nanoGPT/data/cnn_dailymail/val.bin (24.8 MB)
Tokenizing test ...


writing test.bin:   0%|          | 0/64 [00:00<?, ?it/s]

test: 10,738,713 tokens


writing test.bin: 100%|██████████| 64/64 [00:04<00:00, 15.12it/s]


Wrote /kaggle/working/nanoGPT/data/cnn_dailymail/test.bin (21.5 MB)
Done.

Files in data dir:
  prepare.py                0.0 MB
  test.bin                 21.5 MB
  train.bin               540.8 MB
  val.bin                  24.8 MB


## 5. Training config

We write a nanoGPT config file. Key choices:

- `init_from = "scratch"` — random weights.
- `n_layer=6, n_head=6, n_embd=384, block_size=512` — small enough to converge in ~6h on a single T4.
- `batch_size=12, gradient_accumulation_steps=4` → effective batch 48 sequences (24,576 tokens) per step.
- `max_iters=25000` — about one full pass over the ~200M-token training stream.
- `learning_rate=3e-4` with cosine decay to `3e-5` — appropriate for from-scratch training (much higher than the 5e-5 we used for fine-tuning).
- `dtype="float16"` — T4 has no native bfloat16; fp16 is the right choice here.
- `compile=False` — `torch.compile` is unreliable on Colab's T4 driver stack.

In [6]:
config_code = f"""
out_dir = {OUT_DIR!r}
eval_interval = 500
log_interval = 50
eval_iters = 100
eval_only = False
always_save_checkpoint = False
init_from = "scratch"

wandb_log = False
wandb_project = "nanogpt-cnndm"
wandb_run_name = "scratch-6L-384d"

dataset = "cnn_dailymail"
gradient_accumulation_steps = 4
batch_size = 12
block_size = 512

n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.1
bias = False

learning_rate = 3e-4
max_iters = 25000
weight_decay = 0.1
beta1 = 0.9
beta2 = 0.95
grad_clip = 1.0

decay_lr = True
warmup_iters = 200
lr_decay_iters = 25000
min_lr = 3e-5

backend = "nccl"
device = "cuda"
dtype = "float16"
compile = False
"""

config_path = os.path.join(NANO_DIR, "config", "train_cnn_dailymail.py")
with open(config_path, "w") as f:
    f.write(config_code)
print(f"Wrote {config_path}")

Wrote /kaggle/working/nanoGPT/config/train_cnn_dailymail.py


## 6. Train

This is the long-running cell. Output streams live; you'll see lines like:

```
iter 0: loss 10.8345, time 1234.56ms, mfu -100.00%
step 500: train loss 4.21, val loss 4.34
...
```

Watch for **val loss going down**. If after iter 2000 val loss is still above 4.5, something's wrong — paste a screenshot.

The cell auto-detects GPU count: 1 GPU → `python`, 2+ GPUs → `torchrun` (uses all of them).

In [7]:
import torch, subprocess, sys, os
n_gpus = torch.cuda.device_count()
print(f"Detected {n_gpus} GPU(s)")

os.chdir(NANO_DIR)
if n_gpus > 1:
    cmd = ["torchrun", "--standalone", f"--nproc_per_node={n_gpus}", "train.py", "config/train_cnn_dailymail.py"]
else:
    cmd = [sys.executable, "train.py", "config/train_cnn_dailymail.py"]

print("Running:", " ".join(cmd))
print("=" * 80)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()
print(f"\nTraining exit code: {process.returncode}")
assert process.returncode == 0, 'Training failed'

Detected 2 GPU(s)
Running: torchrun --standalone --nproc_per_node=2 train.py config/train_cnn_dailymail.py
W0428 21:55:53.442000 748 torch/distributed/run.py:852] 
W0428 21:55:53.442000 748 torch/distributed/run.py:852] *****************************************
W0428 21:55:53.442000 748 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0428 21:55:53.442000 748 torch/distributed/run.py:852] *****************************************
[W428 21:55:53.175740117 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Overriding config with config/train_cnn_dailymail.py:
Overriding config with config/train_cnn_dailymail.py:

out_dir = '/kaggle/working/nano_out'
eval_interval = 500
log_interval = 50
eval_iters = 100
eval_only = False
always_save_checkpoint = False
init_from = "s

## 7. Inference — load checkpoint and define top-p sampler

nanoGPT's built-in `generate()` only supports temperature + top-k. For a fair comparison with the fine-tuned GPT-2 notebook (which uses top-p / repetition penalty), we hand-write a top-p sampler that calls the model's forward pass directly.

In [8]:
import torch, sys, os
sys.path.insert(0, NANO_DIR)
from model import GPTConfig, GPT
import tiktoken

ckpt_path = os.path.join(OUT_DIR, "ckpt.pt")
print(f"Loading checkpoint from {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location="cuda", weights_only=False)
gptconf = GPTConfig(**ckpt["model_args"])
model = GPT(gptconf)
state = ckpt["model"]
unwanted_prefix = "_orig_mod."
for k in list(state.keys()):
    if k.startswith(unwanted_prefix):
        state[k[len(unwanted_prefix):]] = state.pop(k)
model.load_state_dict(state)
model.eval().cuda()
print(f"Model loaded. Best val loss: {ckpt.get('best_val_loss', 'n/a')}")
print(f"Model config: {ckpt['model_args']}")

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token

@torch.no_grad()
def generate_topp(model, ids, max_new_tokens=128, temperature=0.8, top_p=0.92, rep_penalty=1.2):
    block_size = model.config.block_size
    out = ids.clone()
    for _ in range(max_new_tokens):
        idx = out[:, -block_size:]
        logits, _ = model(idx)
        logits = logits[:, -1, :] / temperature
        # repetition penalty
        for tok_id in set(out[0].tolist()):
            if logits[0, tok_id] > 0:
                logits[0, tok_id] /= rep_penalty
            else:
                logits[0, tok_id] *= rep_penalty
        # top-p (nucleus) filtering
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        probs = torch.softmax(sorted_logits, dim=-1)
        cumprobs = torch.cumsum(probs, dim=-1)
        mask = (cumprobs - probs) > top_p
        sorted_logits[mask] = float("-inf")
        probs = torch.softmax(sorted_logits, dim=-1)
        sampled_pos = torch.multinomial(probs, num_samples=1)
        next_tok = sorted_idx.gather(-1, sampled_pos)
        out = torch.cat([out, next_tok], dim=1)
        if next_tok.item() == EOT:
            break
    return out

def summarize(article, max_new_tokens=128):
    block_size = model.config.block_size
    prompt_text = article.strip() + "\n\nTL;DR:\n"
    ids = enc.encode_ordinary(prompt_text)
    # Reserve room for new tokens
    max_prompt = block_size - max_new_tokens
    if len(ids) > max_prompt:
        # Keep the TL;DR suffix; truncate from the front of the article
        suffix_ids = enc.encode_ordinary("\n\nTL;DR:\n")
        ids = ids[:-len(suffix_ids)][-(max_prompt - len(suffix_ids)):] + suffix_ids
    ids_t = torch.tensor([ids], dtype=torch.long).cuda()
    out = generate_topp(model, ids_t, max_new_tokens=max_new_tokens)
    new_ids = out[0, len(ids):].tolist()
    text = enc.decode(new_ids)
    for stopper in ["\n\n", "<|endoftext|>"]:
        if stopper in text:
            text = text.split(stopper)[0]
    return text.strip()

# Quick smoke test
print("\nSmoke test:")
print(summarize("The president announced today that new economic measures will be implemented to combat inflation."))

Loading checkpoint from /kaggle/working/nano_out/ckpt.pt
number of parameters: 29.94M
Model loaded. Best val loss: 3.704195022583008
Model config: {'n_layer': 6, 'n_head': 6, 'n_embd': 384, 'block_size': 512, 'bias': False, 'vocab_size': 50304, 'dropout': 0.1}

Smoke test:
President Obama was elected in September .
He had recently put pressure on Democrats and Republicans .


## 8. ROUGE evaluation on 500 validation articles

Same metric setup as the fine-tuned GPT-2 notebook (Porter stemmer, F-measure). Realistic expectation for a 10M-non-embedding-param model trained from scratch: **ROUGE-L ~12-18**, well below fine-tuned GPT-2 (~28-32). That gap is the point of the comparison.

In [9]:
from rouge_score import rouge_scorer
from datasets import load_dataset
from tqdm import tqdm
import random, statistics

ds_val = load_dataset("cnn_dailymail", "3.0.0", split="validation")
random.seed(42)
indices = random.sample(range(len(ds_val)), 500)

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
agg = {"rouge1": [], "rouge2": [], "rougeL": []}
predictions = []

for i in tqdm(indices, desc="evaluating"):
    pred = summarize(ds_val[i]["article"])
    ref = ds_val[i]["highlights"]
    predictions.append((i, ref, pred))
    s = scorer.score(ref, pred)
    for k in agg:
        agg[k].append(s[k].fmeasure)

print()
print(f"ROUGE-1: {statistics.mean(agg['rouge1'])*100:.2f}")
print(f"ROUGE-2: {statistics.mean(agg['rouge2'])*100:.2f}")
print(f"ROUGE-L: {statistics.mean(agg['rougeL'])*100:.2f}")

# Save predictions for the report
import json
with open(os.path.join(OUT_DIR, "predictions.json"), "w") as f:
    json.dump([{"idx": i, "reference": r, "prediction": p} for i, r, p in predictions], f, indent=2)
print(f"\nSaved 500 predictions to {os.path.join(OUT_DIR, 'predictions.json')}")

evaluating: 100%|██████████| 500/500 [05:37<00:00,  1.48it/s]


ROUGE-1: 14.56
ROUGE-2: 1.39
ROUGE-L: 9.66

Saved 500 predictions to /kaggle/working/nano_out/predictions.json


## 9. Qualitative samples (5 articles with generated summaries)

In [10]:
for idx, ref, pred in predictions[:5]:
    art = ds_val[idx]["article"]
    print("=" * 80)
    print(f"ARTICLE [#{idx}]:", art[:400].replace("\n", " "), "...")
    print()
    print("REFERENCE:", ref)
    print()
    print("GENERATED:", pred)
    print()

ARTICLE [#10476]: Bacon is a classic American breakfast staple that's best served hot off of the grill- or more surprisingly, a gun. Skilled marksman Dustin Ellerman of Texas decided to take a shot at making his own version of the popular food by cooking it on the end of his M16. Ellerman, a competitive shooter and director of a Christian camp called His way, is also known for winning the third season of the Histor ...

REFERENCE: Dustin Ellerman of Texas decided to take a shot at making his own version of bacon by cooking it on the end of his M16 .
Ellerman is known for winning the third season of the History Channel's shooting competition Top Shot .
Ellerman wraps the bacon in tin foil at the end of the gun and it cooks has he shoots .

GENERATED: The annual K3 movie stars Tom Fiskoff, Allen Ford, Justin Timberlake and David Rothley in their fight .
They are the 'couple' but the mother only wants the show to play with her friend .

ARTICLE [#1824]: The Australian businessman charged 

## 10. Outputs to download

After the run completes, the following files are persisted under the `nano_out/` directory and will survive the session (Kaggle: `/kaggle/working/nano_out/`; Colab: `/content/nano_out/` — download before disconnect):

- `ckpt.pt` — best checkpoint
- `predictions.json` — 500 (reference, prediction) pairs for the report

The training log printed above contains all per-iter loss values for the report's training-curve plot.